In [1]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA
import numpy as np

c:\Users\wr\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\wr\AppData\Local\Programs\Python\Python312\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import pandas as pd
from backtesting.test import SMA
import numpy as np

In [3]:
#读取文件
bu = pd.read_csv('../data/bu.csv')
jd = pd.read_csv('../data/jd.csv')
l = pd.read_csv('../data/l.csv')
pp = pd.read_csv('../data/pp.csv')
ru = pd.read_csv('../data/ru.csv')
v = pd.read_csv('../data/v.csv')

In [4]:
#重命名各列
def column_rename(df):
    df=df.rename(columns={
        df.columns[0]: 'Date',
        df.columns[1]: 'Open',
        df.columns[2]: 'High',
        df.columns[3]: 'Low',
        df.columns[4]: 'Close',
        df.columns[5]: 'Volume'
    }).drop(columns=[df.columns[6], df.columns[7]])
    df['Date'] = pd.to_datetime(df['Date'])
    return df.set_index('Date')
    
new_bu = column_rename(bu)
new_jd = column_rename(jd)
new_l = column_rename(l)
new_pp = column_rename(pp)
new_ru = column_rename(ru)
new_v = column_rename(v)

In [5]:
#ATR计算函数
def ATR(df, period=14):
    high_low = df['High'] - df['Low']
    high_close = abs(df['High'] - df['Close'].shift())
    low_close = abs(df['Low'] - df['Close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


In [6]:
# 尝试提前操作、在当日买卖（6）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    sma_filter = 50
    
    def init(self):
        # 基础均线
        price = self.data.Close
        self.ma0 = self.I(SMA, price, 1)
        self.ma1 = self.I(SMA, price, 10)
        self.ma2 = self.I(SMA, price, 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        # 趋势过滤指标
        self.filter = self.I(SMA, price, self.sma_filter)
        self.entry_price = 0
        self.stop_loss = 0
        self.current_date = self.data.df.index[0]
        self.cross1 = self.current_date
        self.cross2 = self.current_date
        self.fall1 = self.current_date
        self.fall2 = self.current_date

    def next(self):
        current_close = self.data.Close[-1]
        self.current_date = self.data.df.index[-1]
        self.cross1 = self.current_date if crossover(self.ma0, self.ma1) else self.cross1
        self.cross2 = self.current_date if crossover(self.ma0, self.ma2) else self.cross2
        self.fall1 = self.current_date if crossover(self.ma1, self.ma0) else self.fall1
        self.fall2 = self.current_date if crossover(self.ma2, self.ma0) else self.fall2
        below_ma50 = self.data.Close[-1] < self.filter[-1]
        if self.entry_price ==0:
            buy = False
            if self.cross2==self.current_date and below_ma50 and max(self.cross1,self.fall1,self.fall2)==self.cross1 and below_ma50:
                buy = True
            if buy:
                self.buy()
                self.entry_price = current_close  # 记录入场价格
                self.stop_loss = current_close - self.atr[-1] * self.atr_multiplier  # 初始止损位
        
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, current_close - self.atr[-1] * self.atr_multiplier)
            if 0.95 * self.entry_price >= current_close:
                sell = True
            if current_close < self.stop_loss :
                sell = True
            elif self.fall2==self.current_date and max(self.cross1,self.fall1,self.cross2)==self.fall1 and not below_ma50:
                sell = True
            if sell:
                self.sell()
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0  # 重置止损位


In [7]:
# 添加SD计算函数
def SD(price, window=20):
    price = pd.Series(price)
    variance = price.rolling(window).var()
    sd = np.sqrt(variance)
    return sd

In [8]:
# 添加布林带指标（7）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    
    def init(self):
        # 基础均线
        self.price = self.data.Close
        self.ma1 = self.I(SMA, self.price , 10)
        self.ma2 = self.I(SMA, self.price , 20)
        self.sd2 = self.I(SD, self.price , 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        self.upper_band = self.I(lambda close: self.ma2 + self.sd2 * 2, self.data.Close)
        self.lower_band = self.I(lambda close: self.ma2 - self.sd2 * 2, self.data.Close)
        self.band_width = self.I(lambda close: self.upper_band - self.lower_band, self.data.Close)
        self.band_ma5 = self.I(SMA, self.band_width, 5)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            buy = False
            if self.price[-1]>self.ma1[-1] and  self.ma1[-1] > self.ma2[-1]\
            and self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]>self.band_width[-2]:
                buy = True
            if crossover(self.price[-1], self.lower_band[-1]) :
                buy = True
            if buy:
                self.entry_price = self.price[-1]  # 记录入场价格
                self.stop_loss = self.price[-1] - self.atr[-1] * self.atr_multiplier
                self.buy()
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, self.price[-1] - self.atr[-1] * self.atr_multiplier)
            if crossover(self.ma2, self.ma1) or self.price[-1] < self.stop_loss:
                sell = True
            if sell:
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
                self.position.close()

In [9]:
# 放宽条件，让v参与交易（8）
class SmaCross(Strategy):
    atr_period = 14
    atr_multiplier = 1.5
    
    def init(self):
        # 基础均线
        self.price = self.data.Close
        self.ma1 = self.I(SMA, self.price , 10)
        self.ma2 = self.I(SMA, self.price , 20)
        self.sd2 = self.I(SD, self.price , 20)
        # 波动性指标
        self.atr = self.I(ATR, self.data.df, self.atr_period)
        self.upper_band = self.I(lambda close: self.ma2 + self.sd2 * 2, self.data.Close)
        self.lower_band = self.I(lambda close: self.ma2 - self.sd2 * 2, self.data.Close)
        self.band_width = self.I(lambda close: self.upper_band - self.lower_band, self.data.Close)
        self.band_ma5 = self.I(SMA, self.band_width, 5)
        
        self.entry_price = 0
        self.stop_loss = 0

    def next(self):
        
        # 修改买入条件：添加持仓状态检查
        if self.entry_price ==0:
            buy = False
            if self.price[-1]>self.ma1[-1] and  self.ma1[-1] > self.ma2[-1]\
            and self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]>self.band_width[-2]\
                and self.price[-1] < self.upper_band[-1]:
                buy = True
            if crossover(self.price[-1], self.lower_band[-1]) :
                buy = True
            if self.band_width[-1] > self.band_ma5[-1] and self.band_width[-1]<self.band_width[-2]:
                buy = True
            if buy:
                self.entry_price = self.price[-1]  # 记录入场价格
                self.stop_loss = self.price[-1] - self.atr[-1] * self.atr_multiplier
                self.buy()
        
        # 强化卖出条件：添加持仓状态检查
        elif self.entry_price != 0:
            sell = False
            self.stop_loss = min(self.stop_loss, self.price[-1] - self.atr[-1] * self.atr_multiplier)
            if crossover(self.ma2, self.ma1) or self.price[-1] < self.stop_loss:
                sell = True
            if sell:
                self.entry_price = 0  # 重置入场价格
                self.stop_loss = 0    # 重置止损位
                self.position.close()

In [10]:
bt = Backtest(new_bu, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
print(stats)

Backtest.run:   0%|          | 0/945 [00:00<?, ?bar/s]

Start                     2021-06-28 00:00:00
End                       2025-06-26 00:00:00
Duration                   1459 days 00:00:00
Exposure Time [%]                    73.78741
Equity Final [$]                     80110.82
Equity Peak [$]                     133748.96
Commissions [$]                      8467.832
Return [%]                          -19.88918
Buy & Hold Return [%]                 8.56185
Return (Ann.) [%]                    -5.60397
Volatility (Ann.) [%]                21.76174
CAGR [%]                             -3.75782
Sharpe Ratio                         -0.25751
Sortino Ratio                        -0.33784
Calmar Ratio                         -0.12403
Alpha [%]                           -26.20056
Beta                                  0.73715
Max. Drawdown [%]                   -45.18352
Avg. Drawdown [%]                   -10.94842
Max. Drawdown Duration     1205 days 00:00:00
Avg. Drawdown Duration      173 days 00:00:00
# Trades                          

In [11]:
bt = Backtest(new_jd, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-06-28 00:00:00
End                       2025-06-26 00:00:00
Duration                   1459 days 00:00:00
Exposure Time [%]                    69.96904
Equity Final [$]                    98511.272
Equity Peak [$]                    114730.022
Commissions [$]                      8812.076
Return [%]                           -1.48873
Buy & Hold Return [%]               -22.60737
Return (Ann.) [%]                    -0.38931
Volatility (Ann.) [%]                22.19438
CAGR [%]                             -0.25873
Sharpe Ratio                         -0.01754
Sortino Ratio                        -0.03568
Calmar Ratio                         -0.01222
Alpha [%]                              15.814
Beta                                  0.76536
Max. Drawdown [%]                   -31.85209
Avg. Drawdown [%]                    -7.78145
Max. Drawdown Duration     1175 days 00:00:00
Avg. Drawdown Duration      232 days 00:00:00
# Trades                          

In [12]:
bt = Backtest(new_l, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-06-28 00:00:00
End                       2025-06-26 00:00:00
Duration                   1459 days 00:00:00
Exposure Time [%]                    67.49226
Equity Final [$]                    82772.512
Equity Peak [$]                     104126.32
Commissions [$]                       8278.41
Return [%]                          -17.22749
Buy & Hold Return [%]               -12.10114
Return (Ann.) [%]                    -4.79817
Volatility (Ann.) [%]                12.06418
CAGR [%]                             -3.21296
Sharpe Ratio                         -0.39772
Sortino Ratio                        -0.52496
Calmar Ratio                         -0.20577
Alpha [%]                            -9.79409
Beta                                  0.61427
Max. Drawdown [%]                   -23.31765
Avg. Drawdown [%]                    -7.91664
Max. Drawdown Duration     1409 days 00:00:00
Avg. Drawdown Duration      472 days 00:00:00
# Trades                          

In [13]:
bt = Backtest(new_pp, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-06-28 00:00:00
End                       2025-06-26 00:00:00
Duration                   1459 days 00:00:00
Exposure Time [%]                    67.69866
Equity Final [$]                    87422.396
Equity Peak [$]                    115768.984
Commissions [$]                      9612.468
Return [%]                           -12.5776
Buy & Hold Return [%]                -17.1272
Return (Ann.) [%]                    -3.43532
Volatility (Ann.) [%]                13.15928
CAGR [%]                             -2.29495
Sharpe Ratio                         -0.26106
Sortino Ratio                        -0.36853
Calmar Ratio                         -0.13831
Alpha [%]                            -0.27852
Beta                                   0.7181
Max. Drawdown [%]                    -24.8379
Avg. Drawdown [%]                    -5.75774
Max. Drawdown Duration     1354 days 00:00:00
Avg. Drawdown Duration      237 days 00:00:00
# Trades                          

In [14]:
bt = Backtest(new_ru, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Start                     2021-06-28 00:00:00
End                       2025-06-26 00:00:00
Duration                   1459 days 00:00:00
Exposure Time [%]                    69.04025
Equity Final [$]                    102798.88
Equity Peak [$]                     141924.19
Commissions [$]                       8977.85
Return [%]                            2.79888
Buy & Hold Return [%]                 5.01122
Return (Ann.) [%]                     0.72046
Volatility (Ann.) [%]                19.32507
CAGR [%]                              0.47792
Sharpe Ratio                          0.03728
Sortino Ratio                         0.05853
Calmar Ratio                          0.02417
Alpha [%]                             -0.8768
Beta                                  0.73349
Max. Drawdown [%]                   -29.81191
Avg. Drawdown [%]                    -6.55401
Max. Drawdown Duration     1024 days 00:00:00
Avg. Drawdown Duration      117 days 00:00:00
# Trades                          

In [15]:
bt = Backtest(new_v, SmaCross, commission=.002,cash = 100000,
              exclusive_orders=True,trade_on_close=True)
stats = bt.run()
bt.plot()
stats

Backtest.run:   0%|          | 0/218 [00:00<?, ?bar/s]

Start                     2021-09-15 00:00:00
End                       2022-09-15 00:00:00
Duration                    365 days 00:00:00
Exposure Time [%]                    34.29752
Equity Final [$]                    85640.284
Equity Peak [$]                    112149.568
Commissions [$]                       1923.15
Return [%]                          -14.35972
Buy & Hold Return [%]                -26.5634
Return (Ann.) [%]                   -14.90654
Volatility (Ann.) [%]                15.44605
CAGR [%]                            -10.14956
Sharpe Ratio                         -0.96507
Sortino Ratio                        -1.16031
Calmar Ratio                         -0.63063
Alpha [%]                            -6.79104
Beta                                  0.28493
Max. Drawdown [%]                   -23.63744
Avg. Drawdown [%]                   -11.09676
Max. Drawdown Duration      217 days 00:00:00
Avg. Drawdown Duration      103 days 00:00:00
# Trades                          